# Step 13: backend API

Canonical FastAPI source is in the cells tagged `api`. Generate the ignored `.runtime/api.py` entrypoint below, then launch Uvicorn as documented in API_CONTRACT.md. This notebook does not build or start a front end.

Read the server-controlled registry, raw nodes/arcs/track and shared feature definitions. Compute predictions before loading optional server-only reference metadata. The response contains observed GPS, predicted segments, geometry, probabilities and aggregate metrics; reference routes and per-point labels are never returned.

In [1]:
from pathlib import Path
from functools import lru_cache
from time import perf_counter
import json
import os
import re
import numpy as np
import pandas as pd
import shapely
from shapely.strtree import STRtree
from pyproj import Transformer
import xgboost as xgb
import joblib
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, ConfigDict, Field
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support

DEFAULT_ROOT = Path(__file__).resolve().parent.parent if "__file__" in globals() else Path.cwd()
ROOT = Path(os.getenv("PATHSNAP_PROJECT_ROOT", str(DEFAULT_ROOT))).resolve()
DATA_ROOT = Path(os.getenv("PATHSNAP_DATA_ROOT", str(ROOT / "Student folder"))).resolve()
PARTITION = os.getenv("PATHSNAP_PARTITION", "student_test")
if PARTITION not in {"train", "student_test", "evaluator_test"}:
    raise RuntimeError("Unsupported server partition.")
ENABLE_METRICS = os.getenv("PATHSNAP_ENABLE_METRICS", "1") == "1"
ORIGINS = [value.strip() for value in os.getenv("PATHSNAP_CORS_ORIGINS",
    "http://localhost:3000,http://127.0.0.1:3000,http://localhost:5173,http://127.0.0.1:5173").split(",") if value.strip()]


def notebook_definitions(filename):
    namespace = {}
    notebook = json.loads((ROOT / filename).read_text())
    for cell in notebook["cells"]:
        if cell["cell_type"] == "code":
            source = "".join(cell["source"])
            exec(compile(source, filename, "exec"), namespace)
    return namespace


FEATURES = notebook_definitions("05_feature_helpers.ipynb")
DIRECTION = notebook_definitions("05_direction_features.ipynb")
HMM = notebook_definitions("08_hmm_helpers.ipynb")


def registry():
    try:
        return json.loads((ROOT / "models/registry.json").read_text())
    except (OSError, json.JSONDecodeError):
        raise HTTPException(503, "Model registry is unavailable.") from None


def model_entry(model_id):
    entry = registry()["models"].get(model_id)
    if entry is None:
        raise HTTPException(404, "Unknown model.")
    if entry.get("status") != "trained" or not entry.get("model_path") or not (ROOT / entry["model_path"]).is_file():
        raise HTTPException(503, "This model is not available yet.")
    return entry


@lru_cache(maxsize=8)
def read_model(path, modified_ns, kind):
    if kind == "xgboost_ranker":
        model = xgb.XGBRanker(n_jobs=2)
        model.load_model(path)
        return {"model": model}
    saved = joblib.load(path)
    return saved if isinstance(saved, dict) else {"model": saved}


def softmax(values, groups):
    values = np.asarray(values, dtype=float)
    weights = np.exp(values - pd.Series(values).groupby(groups, sort=False).transform("max").to_numpy())
    return weights / pd.Series(weights).groupby(groups, sort=False).transform("sum").to_numpy()


def positive_probability(model, matrix):
    classes = np.asarray(model.classes_)
    positions = np.flatnonzero(classes == 1)
    if not len(positions):
        return np.zeros(len(matrix))
    return model.predict_proba(matrix)[:, positions[0]]


def model_probability(model_id, features, ancestors=()):
    if model_id in ancestors:
        raise HTTPException(503, "Invalid stacking dependency cycle.")
    entry = model_entry(model_id)
    path = ROOT / entry["model_path"]
    saved = read_model(str(path), path.stat().st_mtime_ns, entry["kind"])
    model = saved.get("model", saved.get("pipeline", saved.get("combiner")))
    if model is None:
        raise HTTPException(503, "Model bundle has no estimator.")
    saved_columns = saved.get("meta_feature_columns") if entry["kind"] == "stacking" else saved.get("feature_columns")
    if entry.get("feature_columns_path"):
        columns = json.loads((ROOT / entry["feature_columns_path"]).read_text())
        if isinstance(columns, dict):
            columns = columns["feature_columns"]
    elif saved_columns is not None:
        columns = list(saved_columns)
    else:
        raise HTTPException(503, "Model has no registered feature order.")
    if saved_columns is not None and list(saved_columns) != list(columns):
        raise HTTPException(503, "Model bundle and registry feature order disagree.")
    groups = features["group_id"].to_numpy()
    if entry["kind"] == "stacking":
        order = entry["base_model_order"]
        matrix = pd.DataFrame({name: model_probability(name, features, ancestors + (model_id,)) for name in order})
        matrix["distance"] = features["distance"].to_numpy()
        matrix["candidate_count"] = features["candidate_count"].to_numpy()
        matrix["n_candidates"] = matrix["candidate_count"]
    else:
        matrix = features
    if not set(columns).issubset(matrix.columns):
        raise HTTPException(503, "Required model features are unavailable.")
    matrix = matrix[columns]
    if saved.get("scaler") is not None and not hasattr(model, "named_steps"):
        matrix = saved["scaler"].transform(matrix)
    if entry["kind"] == "xgboost_ranker":
        if model.get_booster().feature_names != list(columns):
            raise HTTPException(503, "Ranker feature order does not match the registry.")
        probability = softmax(model.predict(matrix), groups)
    elif entry["kind"] == "stacking" and hasattr(model, "decision_function"):
        probability = softmax(model.decision_function(matrix), groups)
    else:
        values = positive_probability(model, matrix)
        totals = pd.Series(values).groupby(groups, sort=False).transform("sum").to_numpy()
        sizes = features.groupby("group_id", sort=False)["segment_id"].transform("size").to_numpy()
        probability = np.divide(values, totals, out=1.0 / sizes, where=totals > 0)
    if not np.isfinite(probability).all():
        raise HTTPException(503, "Model produced non-finite probabilities.")
    return probability


In [2]:
@lru_cache(maxsize=4)
def prepare_trip(trip_id):
    folder = DATA_ROOT / PARTITION / trip_id
    paths = {extension: folder / f"{trip_id}.{extension}" for extension in ["nodes", "arcs", "track"]}
    if not all(path.is_file() for path in paths.values()):
        raise HTTPException(404, "Trip not found in the server partition.")
    nodes = pd.read_csv(paths["nodes"], sep=r"\s+", header=None).to_numpy()
    arcs = pd.read_csv(paths["arcs"], sep=r"\s+", header=None).to_numpy(dtype=int)
    track = pd.read_csv(paths["track"], sep=r"\s+", header=None, names=["lon", "lat", "time"])
    if track.empty or not np.isfinite(track.to_numpy()).all():
        raise HTTPException(422, "Trip contains invalid GPS observations.")
    zone = int((track["lon"].mean() + 180) // 6) + 1
    epsg = (32600 if track["lat"].mean() >= 0 else 32700) + zone
    projection = Transformer.from_crs(4326, epsg, always_xy=True)
    xy = np.column_stack(projection.transform(nodes[:, 0], nodes[:, 1]))
    gps_xy = np.column_stack(projection.transform(track["lon"], track["lat"]))
    segments = shapely.linestrings(xy[arcs])
    gps = shapely.points(gps_xy)
    settings = json.loads((ROOT / "outputs/candidates/settings.json").read_text())
    p, s = STRtree(segments).query(gps, predicate="dwithin", distance=settings["radius_m"])
    candidates = pd.DataFrame({"point_id": p, "segment_id": s,
        "distance": shapely.distance(gps[p], segments[s])}).sort_values(["point_id", "distance", "segment_id"]).reset_index(drop=True)
    candidates = candidates[candidates.groupby("point_id").cumcount() < settings["K"]].copy()
    candidates["trip_id"], candidates["group_id"] = trip_id, candidates["point_id"]
    candidates["label"] = pd.array([pd.NA] * len(candidates), dtype="Int8")
    candidates = candidates[["trip_id", "group_id", "point_id", "segment_id", "distance", "label"]]
    features = FEATURES["add_features"](candidates, xy, arcs, gps_xy, track["time"].to_numpy())
    extra = DIRECTION["direction_features"](features, xy, arcs, gps_xy, track["time"].to_numpy())
    features[DIRECTION["direction_columns"]] = extra
    nearest = features["distance"] - features["distance_above_nearest_m"]
    features["distance_ratio"] = ((features["distance"] + 1) / (nearest + 1)).astype("float32")
    return {"nodes": nodes, "arcs": arcs, "track": track, "features": features, "segments": segments, "gps": gps}


@lru_cache(maxsize=4)
def trip_network(trip_id):
    return HMM["load_network"](trip_id, partition=PARTITION, data_root=str(DATA_ROOT))


def aligned_reference(trip_id, prepared):
    track = prepared["track"]
    stored = ROOT / "outputs/student_test/points.csv"
    if PARTITION == "student_test" and DATA_ROOT == (ROOT / "Student folder").resolve() and stored.is_file():
        reference = pd.read_csv(stored, dtype={"trip_id": str})
        reference = reference[reference["trip_id"] == trip_id].sort_values("point_id")
        if len(reference) == len(track) and np.allclose(reference[["lon", "lat", "time"]], track[["lon", "lat", "time"]], rtol=0, atol=1e-9):
            return reference[["point_id", "true_arc_id", "bad_gps"]].reset_index(drop=True)
    path = DATA_ROOT / PARTITION / trip_id / f"{trip_id}.route"
    if not path.is_file():
        return None
    try:
        route = pd.read_csv(path, sep=r"\s+", header=None)[0].to_numpy(dtype=int)
    except (pd.errors.EmptyDataError, ValueError):
        return None
    if not len(route) or route.min() < 0 or route.max() >= len(prepared["segments"]):
        return None
    D = shapely.distance(prepared["gps"][:, None], prepared["segments"][route][None, :])
    n, r = D.shape
    parents = np.zeros((n, r), dtype=np.int32)
    previous = np.minimum(D[0], 150)
    for i in range(1, n):
        prefix = np.minimum.accumulate(previous)
        improved = np.r_[True, previous[1:] < prefix[:-1]]
        parents[i] = np.maximum.accumulate(np.where(improved, np.arange(r), 0))
        previous = np.minimum(D[i], 150) + prefix
    position = np.zeros(n, dtype=np.int32)
    position[-1] = previous.argmin()
    for i in range(n - 1, 0, -1):
        position[i - 1] = parents[i, position[i]]
    threshold = float(json.loads((ROOT / "outputs/folds/contract.json").read_text())["point_flag_threshold_m"])
    bad = D[np.arange(n), position] > threshold
    return pd.DataFrame({"point_id": np.arange(n), "true_arc_id": np.where(bad, -1, route[position]), "bad_gps": bad})


def aggregate_metrics(trip_id, prepared, scores, selected):
    reference = aligned_reference(trip_id, prepared)
    if reference is None:
        return None
    valid = reference[~reference["bad_gps"]]
    rows = scores.merge(valid, on="point_id", validate="many_to_one")
    rows["target"] = rows["segment_id"].eq(rows["true_arc_id"]).astype("int8")
    chosen = selected[["point_id", "segment_id"]].assign(predicted=1)
    rows = rows.merge(chosen, on=["point_id", "segment_id"], how="left", validate="one_to_one")
    predicted = rows["predicted"].fillna(0).astype("int8")
    precision, recall, f1, _ = precision_recall_fscore_support(rows["target"], predicted, labels=[0, 1], zero_division=0)
    matched = valid.merge(selected[["point_id", "segment_id"]], on="point_id", how="left", validate="one_to_one")
    positive = rows.loc[rows["target"] == 1, "score"]
    available = selected.set_index("point_id")["segment_id"].reindex(range(len(prepared["track"])), fill_value=-1).to_numpy()
    network = trip_network(trip_id)
    disconnected = sum(not HMM["transition_is_reachable"](network, int(a), int(b)) for a, b in zip(available[:-1], available[1:]) if min(a, b) >= 0)
    missing = int(sum(min(a, b) < 0 for a, b in zip(available[:-1], available[1:])))
    return {"scored_points": len(valid), "excluded_points": int(reference["bad_gps"].sum()),
        "top1": float(matched["segment_id"].eq(matched["true_arc_id"]).sum() / len(valid)) if len(valid) else None,
        "macro_f1": float(f1.mean()), "precision": float(precision[1]), "recall": float(recall[1]),
        "conditional_log_loss": float(-np.log(positive.clip(lower=1e-15)).mean()) if len(positive) else None,
        "candidate_misses": int(len(valid) - len(positive)), "candidate_coverage": float(len(positive) / len(valid)) if len(valid) else None,
        "confusion_matrix": confusion_matrix(rows["target"], predicted, labels=[0, 1]).tolist(),
        "disconnected_jumps": int(disconnected), "missing_transition_pairs": missing}


In [3]:
app = FastAPI(title="PathSnap API", version="1.0.0")
app.add_middleware(CORSMiddleware, allow_origins=ORIGINS, allow_credentials=False,
                   allow_methods=["GET", "POST"], allow_headers=["Content-Type"])


class PredictRequest(BaseModel):
    model_config = ConfigDict(extra="forbid")
    trip_id: str = Field(pattern=r"^[0-9]{8}$")
    model: str | None = Field(default=None, pattern=r"^[a-zA-Z0-9_]+$")
    decode: bool = False


@app.get("/health")
def health():
    state = registry()
    ready = [name for name, entry in state["models"].items() if entry.get("status") == "trained" and entry.get("model_path") and (ROOT / entry["model_path"]).is_file()]
    return {"status": "ok" if state["active_model"] in ready else "degraded", "active_model": state["active_model"],
            "ready_models": ready, "partition": PARTITION, "api_version": "1.0.0"}


@app.get("/models")
def models():
    state = registry()
    return {"active_model": state["active_model"], "models": [
        {"id": name, "status": entry["status"], "kind": entry["kind"],
         "ready": entry.get("status") == "trained" and bool(entry.get("model_path")) and (ROOT / entry["model_path"]).is_file(),
         "decoder_available": bool(entry.get("hmm_config_path")) and (ROOT / entry["hmm_config_path"]).is_file()}
        for name, entry in state["models"].items()]}


@app.get("/trips")
def trips():
    folder = DATA_ROOT / PARTITION
    ids = sorted(path.name for path in folder.iterdir() if path.is_dir() and re.fullmatch(r"[0-9]{8}", path.name)
                 and all((path / f"{path.name}.{suffix}").is_file() for suffix in ["nodes", "arcs", "track"])) if folder.is_dir() else []
    return {"partition": PARTITION, "trips": [{"trip_id": trip_id} for trip_id in ids], "count": len(ids)}


@app.post("/predict")
def predict(request: PredictRequest):
    started = perf_counter()
    model_id = request.model or registry()["active_model"]
    entry = model_entry(model_id)
    if request.decode and not entry.get("hmm_config_path"):
        raise HTTPException(409, "No decoder configuration is registered for this model.")
    hmm = None
    if request.decode:
        path = ROOT / entry["hmm_config_path"]
        if not path.is_file():
            raise HTTPException(409, "The registered decoder configuration is unavailable.")
        hmm = json.loads(path.read_text())
        if hmm.get("tuning_source") and entry.get("oof_path") and (ROOT / hmm["tuning_source"]).resolve() != (ROOT / entry["oof_path"]).resolve():
            raise HTTPException(409, "Decoder configuration belongs to a different model.")
    start = perf_counter()
    prepared = prepare_trip(request.trip_id)
    features = prepared["features"]
    features_s = perf_counter() - start
    start = perf_counter()
    probability = model_probability(model_id, features) if len(features) else np.array([])
    model_s = perf_counter() - start
    scores = features[["point_id", "group_id", "segment_id"]].copy()
    scores["score"] = probability
    ordered = scores.sort_values(["group_id", "segment_id"]).reset_index(drop=True)
    selected = ordered.loc[ordered.groupby("group_id")["score"].idxmax(), ["point_id", "group_id", "segment_id"]].copy()
    decode_s = 0.0
    if request.decode and len(scores):
        start = perf_counter()
        metadata = pd.DataFrame({"point_id": np.arange(len(prepared["track"])), "group_id": np.arange(len(prepared["track"]))})
        state = HMM["prepare_trip"](scores, metadata, trip_network(request.trip_id), gap_reset_s=hmm.get("gap_reset_s", 30))
        selected, posterior = HMM["decode_trip"](state, hmm["beta"], hmm["emission_weight"], posterior=True)
        scores = scores.drop(columns="score").merge(posterior, on=["group_id", "segment_id"], validate="one_to_one")
        decode_s = perf_counter() - start
    start = perf_counter()
    metrics = aggregate_metrics(request.trip_id, prepared, scores, selected) if ENABLE_METRICS else None
    evaluation_s = perf_counter() - start
    start = perf_counter()
    scored_selection = selected.merge(scores[["point_id", "segment_id", "score"]], on=["point_id", "segment_id"], how="left", validate="one_to_one").set_index("point_id")
    nodes, arcs = prepared["nodes"], prepared["arcs"]
    predictions = []
    for point_id in range(len(prepared["track"])):
        if point_id not in scored_selection.index:
            predictions.append({"point_id": point_id, "segment_id": None, "probability": None, "coordinates": None})
            continue
        row = scored_selection.loc[point_id]
        segment_id = int(row["segment_id"])
        predictions.append({"point_id": point_id, "segment_id": segment_id, "probability": float(row["score"]),
                            "coordinates": nodes[arcs[segment_id]].tolist()})
    road_network = {"type": "FeatureCollection", "features": [{"type": "Feature", "properties": {"segment_id": int(segment)},
        "geometry": {"type": "LineString", "coordinates": nodes[arcs[segment]].tolist()}} for segment in np.unique(features["segment_id"])]}
    gps = [{"point_id": int(i), "lon": float(row.lon), "lat": float(row.lat), "time": float(row.time)} for i, row in enumerate(prepared["track"].itertuples(index=False))]
    geometry_s = perf_counter() - start
    return {"trip_id": request.trip_id, "model": model_id, "partition": PARTITION, "decode_applied": request.decode,
            "probability_kind": "HMM posterior" if request.decode else "per-point model emission",
            "point_count": len(gps), "candidate_rows": len(features), "gps": gps, "predictions": predictions,
            "road_network": road_network, "metrics": metrics, "metrics_status": "available" if metrics is not None else "unavailable_or_disabled",
            "timings": {"features_s": features_s, "model_s": model_s, "decode_s": decode_s,
                "evaluation_s": evaluation_s, "geometry_s": geometry_s, "server_processing_s": perf_counter() - started}}


## Generate the CLI entrypoint

The notebook remains the source of truth. Regenerate after editing tagged cells. `.runtime/.gitignore` ignores generated files locally; do not hand-edit the generated Python module.

In [4]:
import nbformat
runtime = ROOT / ".runtime"
runtime.mkdir(parents=True, exist_ok=True)
(runtime / ".gitignore").write_text("*\n")
source_notebook = nbformat.read(ROOT / "13_backend_api.ipynb", as_version=4)
source = "\n\n".join(cell.source for cell in source_notebook.cells
                      if cell.cell_type == "code" and "api" in cell.metadata.get("tags", []))
compile(source, ".runtime/api.py", "exec")
(runtime / "api.py").write_text(source + "\n")
print("Generated", runtime / "api.py")

Generated /Users/admin/ml_hack/.runtime/api.py


## In-process API verification

Use the real registry/model and the smallest available student-test trip. Validate metadata endpoints, a complete real prediction, missing-model/trip errors, request path rejection, CORS and the absence of reference/label fields at every response level. This is an ASGI TestClient check, not a claim of LAN deployment.

In [5]:
import runpy
from fastapi.testclient import TestClient
api = runpy.run_path(str(runtime / "api.py"))
client = TestClient(api["app"])
assert client.get("/health").status_code == 200
assert client.get("/models").status_code == 200
available = client.get("/trips").json()["trips"]
assert available
trip_id = min((row["trip_id"] for row in available), key=lambda name: sum(1 for _ in (DATA_ROOT / PARTITION / name / f"{name}.track").open()))
response = client.post("/predict", json={"trip_id": trip_id})
assert response.status_code == 200, response.text
result = response.json()
assert len(result["gps"]) == len(result["predictions"]) == result["point_count"]
assert result["candidate_rows"] > 0 and any(row["segment_id"] is not None for row in result["predictions"])
forbidden = {"label", "labels", "true_arc_id", "route_pos", "bad_gps", "reference_route", "ground_truth"}
def check_keys(value):
    if isinstance(value, dict):
        assert not forbidden.intersection(value)
        for nested in value.values():
            check_keys(nested)
    elif isinstance(value, list):
        for nested in value:
            check_keys(nested)
check_keys(result)
assert client.post("/predict", json={"trip_id": "99999999"}).status_code == 404
assert client.post("/predict", json={"trip_id": trip_id, "model": "missing_model"}).status_code == 404
assert client.post("/predict", json={"trip_id": "../../etc/passwd"}).status_code == 422
assert client.post("/predict", json={"trip_id": trip_id, "partition": "train"}).status_code == 422
if ORIGINS:
    assert client.options("/predict", headers={"Origin": ORIGINS[0], "Access-Control-Request-Method": "POST"}).headers["access-control-allow-origin"] == ORIGINS[0]
print("Verified real prediction:", result["model"], trip_id, "| Points:", result["point_count"], "| Candidate rows:", result["candidate_rows"])
print("Server processing seconds:", result["timings"]["server_processing_s"])
ready = client.get("/models").json()["models"]
for entry in ready:
    if entry["ready"] and entry["id"] != result["model"]:
        other = client.post("/predict", json={"trip_id": trip_id, "model": entry["id"]})
        assert other.status_code == 200, other.text
        check_keys(other.json())
        print("Verified available model:", entry["id"])
active = next(entry for entry in ready if entry["id"] == result["model"])
if active["decoder_available"]:
    decoded = client.post("/predict", json={"trip_id": trip_id, "decode": True})
    assert decoded.status_code == 200, decoded.text
    decoded_result = decoded.json()
    assert decoded_result["decode_applied"] and len(decoded_result["predictions"]) == result["point_count"]
    check_keys(decoded_result)
    print("Verified current-model HMM decode; seconds:", decoded_result["timings"]["decode_s"])
print("All endpoints, available models, error cases, CORS and no-truth-response checks passed. LAN deployment has not been tested.")
namespace = api["predict"].__globals__
saved_enabled = namespace["ENABLE_METRICS"]
saved_reference = namespace["aligned_reference"]
saved_predict = namespace["model_probability"]
events = []
def record_prediction(*args, **kwargs):
    values = saved_predict(*args, **kwargs)
    events.append("prediction")
    return values
def record_reference(*args, **kwargs):
    assert events and events[-1] == "prediction"
    events.append("evaluation")
    return saved_reference(*args, **kwargs)
try:
    namespace["model_probability"] = record_prediction
    namespace["aligned_reference"] = record_reference
    namespace["ENABLE_METRICS"] = False
    no_metrics = client.post("/predict", json={"trip_id": trip_id})
    assert no_metrics.status_code == 200 and no_metrics.json()["metrics"] is None
    assert events == ["prediction"]
    events.clear()
    namespace["ENABLE_METRICS"] = True
    scored = client.post("/predict", json={"trip_id": trip_id})
    assert scored.status_code == 200 and events == ["prediction", "evaluation"]
    check_keys(scored.json())
finally:
    namespace["ENABLE_METRICS"] = saved_enabled
    namespace["aligned_reference"] = saved_reference
    namespace["model_probability"] = saved_predict
print("Verified: reference metadata is read only after prediction and never when metrics are disabled.")

Verified real prediction: xgboost 00000052 | Points: 262 | Candidate rows: 17397
Server processing seconds: 0.7101375799975358


Verified current-model HMM decode; seconds: 1.2755382979812566
All endpoints, available models, error cases, CORS and no-truth-response checks passed. LAN deployment has not been tested.


Verified: reference metadata is read only after prediction and never when metrics are disabled.
